In [ ]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

## Lets load the Boston House Pricing Dataset

In [ ]:
#Fetch the raw data
data_url = "https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv"
raw_df = pd.read_csv(data_url)

#Extract features and target
data = raw_df.drop('medv', axis=1).values
target = raw_df['medv'].values

#Define features names
feature_names = [col.upper() for col in raw_df.columns if col != 'medv']



### Data Set Characteristics:

* **Number of Instances:** 506
* **Number of Attributes:** 13 numeric/categorical predictive. The Median Value (attribute 14) is the target.

### Attribute Information (in order):
* **CRIM:** per capita crime rate by town
* **ZN:** proportion of residential land zoned for lots over 25,000 sq.ft.
* **INDUS:** proportion of non-retail business acres per town
* **CHAS:** Charles River dummy variable (= 1 if tract bounds river; 0 otherwise)
* **NOX:** nitric oxides concentration (parts per 10 million)
* **RM:** average number of rooms per dwelling
* **AGE:** proportion of owner-occupied units built prior to 1940
* **DIS:** weighted distances to five Boston employment centres
* **RAD:** index of accessibility to radial highways
* **TAX:** full-value property-tax rate per $10,000
* **PTRATIO:** pupil-teacher ratio by town
* **B:** 1000(Bk - 0.63)^2 where Bk is the proportion of blacks by town
* **LSTAT:** % lower status of the population
* **Price:** Median value of owner-occupied homes in $1000's (Target Variable)
*

In [ ]:
print(data)

In [ ]:
print(target)

In [ ]:
print(feature_names)

## Preparing The Dataset

In [ ]:
#Construct the final dataframe
dataset = pd.DataFrame(data, columns=feature_names)
dataset['Price']= target

In [ ]:
dataset.head()

In [ ]:
dataset.info()

In [ ]:
# Summarizing the stats of the data
dataset.describe()

## check the missing values

In [ ]:
dataset.isnull().sum()

In [ ]:
### Exploratory Data Analysis
## correlation 
dataset.corr()

In [ ]:
import seaborn as sns
sns.pairplot(dataset)

 ## Analyzing The Correlated Features

In [ ]:
dataset.corr()

In [ ]:
plt.scatter(dataset['CRIM'],dataset['Price'])
plt.xlabel("Crime Rate")
plt.ylabel("Price")

In [ ]:
plt.scatter(dataset['RM'],dataset['Price'])
plt.xlabel("RM")
plt.ylabel("Price")

In [ ]:
import seaborn as sns
sns.regplot(x="RM",y="Price",data = dataset)

In [ ]:
sns.regplot(x="LSTAT",y="Price",data=dataset)

In [ ]:
sns.regplot(x="CHAS",y="Price",data=dataset)

In [ ]:
sns.regplot(x="PTRATIO",y="Price",data=dataset)

## Independent and Dependent Features


In [ ]:
X=dataset.iloc[:,:-1]
y=dataset.iloc[:,-1]

In [ ]:
X.head()

In [ ]:
y

In [ ]:
## TRAIN TEST SPLIT
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size =0.3, random_state=42) 

In [ ]:
X_train

In [ ]:
X_test

In [ ]:
## Standardize the dataset
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()

In [ ]:
X_train=scaler.fit_transform(X_train)

In [ ]:
X_test=scaler.transform(X_test)

In [ ]:
X_train

In [ ]:
X_test

# Model Training

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
regression=LinearRegression()

In [ ]:
regression.fit(X_train,y_train)

In [ ]:
## print the coefficient and the intercept
print(regression.coef_)

In [ ]:
print(regression.intercept_)

In [ ]:
## on which parameters the model has been trained
regression.get_params()

In [ ]:
### prediction with test data
reg_pred = regression.predict(X_test)

## Assumption

In [ ]:
## plot a scatter plot for the prediction 
plt.scatter(y_test,reg_pred)

In [ ]:
## Residuals
residuals=y_test-reg_pred

In [ ]:
residuals

In [ ]:
##  Plot this residuals
sns.displot(residuals,kind="kde")

In [ ]:
## Scatter plot with resspect to prediction and residuals
# uniform distribution
plt.scatter(reg_pred,residuals)

In [ ]:
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
print(mean_absolute_error(y_test,reg_pred))
print(mean_squared_error(y_test,reg_pred))
print(np.sqrt(mean_squared_error(y_test,reg_pred)))

## R square and adjusted R square
Formula 
### R^2=1-SSR/SST
R^2 = coefficient of determination, SSR = sum of squares of residuals, SST = total sum of squares 

In [ ]:
from sklearn.metrics import r2_score
score = r2_score(y_test,reg_pred)
print(score)

## Adjusted R2 = 1-[(1-R1)*(n-1)/(n-k-1)]
where:
R2: The R2 of the model, n: The number of observations, k: The number of predictor variables

In [ ]:
# display adjusted R-squared
1 - (1-score)*(len(y_test)-1)/(len(y_test)-X_test.shape[1]-1)

## New Data Prediction


In [ ]:
single_data_point=data[0].reshape(1,-1)

In [ ]:
single_data_df = pd.DataFrame(single_data_point,columns=feature_names)


In [ ]:
# transformation of new data
scaled_data=scaler.transform(single_data_df)
print("Scaled Data\n",scaled_data)

In [ ]:
prediction=regression.predict(scaled_data)
print("\nPredicted Price:\n",prediction)

## Pickling the model file for Deployment

In [ ]:
import pickle

In [ ]:
pickle.dump(regression,open('regmodel.pkl','wb'))

In [ ]:
pickled_model=pickle.load(open('regmodel.pkl','rb'))

In [ ]:
## Prediction 
pickled_model.predict(scaler.transform(single_data_df))